In [1]:
import pandas as pd
import plotly.express as px

def main():
    m = pd.read_csv("data/meter_readings.csv", parse_dates=["timestamp"])
    b = pd.read_csv("data/buildings.csv")

    # Total kWh per 15-min interval
    m["kwh"] = m["power_kw"] * 0.25

    # Building totals
    g = m.groupby(["building_id", pd.Grouper(key="timestamp", freq="D")])["kwh"].sum().reset_index()
    g2 = g.groupby("building_id")["kwh"].agg(["mean", "std", "max", "min"]).reset_index()
    g2 = g2.merge(b[["building_id","building_type","efficiency_grade"]], on="building_id", how="left")

    print("Daily kWh stats (per building):")
    print(g2.head(10))

    # Plot: sample buildings daily kWh
    sample = g["building_id"].drop_duplicates().head(6).tolist()
    fig = px.line(g[g["building_id"].isin(sample)], x="timestamp", y="kwh", color="building_id",
                  title="Daily energy (kWh) - sample buildings")
    fig.show()

    # Plot: appliance share overall
    app = m.groupby("appliance")["kwh"].sum().sort_values(ascending=False).reset_index()
    fig2 = px.bar(app, x="appliance", y="kwh", title="Total kWh by appliance (all buildings)")
    fig2.show()

if __name__ == "__main__":
    main()


Daily kWh stats (per building):
  building_id        mean        std         max         min building_type  \
0        B000   97.835982   5.089458  106.757689   89.426195        Office   
1        B001   68.499630   7.966843   79.694563   54.730392        Office   
2        B002  202.370284  27.659824  236.109495  157.776196   Residential   
3        B003  107.562211   7.157803  118.510608   96.069302   Residential   
4        B004  125.788043   6.134833  136.608692  116.357399        Office   
5        B005  289.719613  45.172907  350.842988  221.240603   Residential   
6        B006  377.437804  24.050864  412.309769  343.653186   Residential   
7        B007  326.596099  49.146743  389.339389  247.764564   Residential   
8        B008  146.335278  20.771091  171.803699  112.103170   Residential   
9        B009   21.362945   1.026175   23.500311   19.725241        Office   

  efficiency_grade  
0                C  
1                B  
2                A  
3                C  
4   